# 06 — Pahang S2 SVM grouped validation and Rubber sample-size experiment

本 notebook 使用 **Pahang excluding Bentong** 的七类 polygon 像元样本，重新训练并验证一套完整 SVM，而不是再次运行冻结的 Bentong 模型。

主实验严格继承 Bentong SVM 的核心设计：

- 每个 polygon 最多保留100个像元；每个 polygon 的总像元权重为1；
- 5-fold outer spatial grouped validation；
- 每个 outer training set 内进行3-fold inner grouped tuning；
- `StandardScaler` 只在训练 split 内拟合；
- 使用与 Bentong 相同的 Linear/RBF SVM 候选和模型选择顺序；
- 像元指标采用等 polygon 总权重，polygon 预测采用平均 decision score；
- 所有慢速阶段按 fold/checkpoint 保存，可在 Colab 中断后续跑。

由于 Pahang CSV 只有最终冻结的 Sentinel-2 predictors，本实验固定使用 **S2 feature set**，不重复比较 S1/DEM feature stacks。

最后增加 Rubber 缩减实验：验证折保持不变，仅把训练侧 Rubber 缩减到接近 Bentong 的10个空间组/36个 polygon 的规模，再与完整 Pahang Rubber 训练进行配对比较。这比单纯比较 Bentong 与 Pahang 的准确率更能检验“Rubber 样本不足”这一猜想。

## 运行说明

1. 在 Colab 使用 **File → Open notebook → Upload** 上传整个 `.ipynb`，不要把 notebook 的 JSON 文本粘贴进代码单元。
2. 确保 Google Drive 中存在：`<DURIAN_DATA_ROOT>/Pahang_without_Bentong_S2_pixel_samples_2025_v1.csv`。
3. 从上到下分步运行。长时间阶段有 checkpoint；断线后重新挂载 Drive 并从头运行，已完成的 fold 会自动复用。
4. RBF-SVC 不使用 GPU，也没有 `N_JOBS=-1`。Pahang 保留像元数比 Bentong 更多，完整运行可能持续数小时；不要在同一 Colab 会话中并行运行其他高内存模型。
5. 主实验完成后才运行 Rubber 缩减实验。若只想先得到 Pahang 内部验证结果，可将 `RUN_RUBBER_ABLATION=False`。

## 0. 环境、路径与锁定参数

### 0.1 挂载 Google Drive

- **作用：** 读取 Pahang CSV，并把 checkpoint、表格、图像和最终模型持续保存到 Drive。

In [ ]:
from pathlib import Path
import os


def _find_repository_root(start: Path) -> Path:
    current = start.resolve()
    while current.parent != current:
        if (current / "README.md").exists() and (current / "code").exists():
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from within the repository tree.")


REPO_ROOT = _find_repository_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("DURIAN_DATA_ROOT", REPO_ROOT / "private_data")
).expanduser().resolve()
REPO_OUTPUT_ROOT = Path(
    os.environ.get("DURIAN_OUTPUT_ROOT", REPO_ROOT / "outputs" / "runs")
).expanduser().resolve()
REPO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Private data root:", DATA_ROOT)
print("Run output root:", REPO_OUTPUT_ROOT)


### 0.2 导入依赖

In [ ]:
import gc
import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

warnings.filterwarnings('once')
sns.set_theme(style='whitegrid', context='notebook')

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('joblib:', joblib.__version__)

### 0.3 输入、输出与运行开关

- **作用：** 锁定正式 CSV 和稳定输出目录。
- `RUN_RUBBER_ABLATION=True` 会在主实验后运行直接的 Rubber 样本规模检验。
- 如需新实验而不是续跑，请修改 `RUN_NAME`，不要删除或混用旧 checkpoint。

In [ ]:
DRIVE_ROOT = DATA_ROOT
INPUT_CSV = DRIVE_ROOT / 'Pahang_without_Bentong_S2_pixel_samples_2025_v1.csv'

RUN_NAME = 'Pahang_S2_SVM_grouped_validation_20260712_run01'
OUTPUT_DIR = REPO_OUTPUT_ROOT / RUN_NAME
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
METADATA_DIR = OUTPUT_DIR / 'metadata'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'

# Optional Bentong outputs used only for the final descriptive comparison.
BENTONG_RF_DIR = DRIVE_ROOT / 'RF_grouped_validation_20260623_153912_UTC'
BENTONG_SVM_DIR = DRIVE_ROOT / 'SVM_grouped_validation_20260624_run01'

RUN_RUBBER_ABLATION = True
RUBBER_ABLATION_REPEATS = 5

for folder in [
    OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, MODEL_DIR,
    METADATA_DIR, CHECKPOINT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')

print('Input CSV:', INPUT_CSV)
print('Output directory:', OUTPUT_DIR)
print('Run Rubber ablation:', RUN_RUBBER_ABLATION)

### 0.4 锁定与 Bentong 一致的类别、S2 predictors、fold 和 SVM candidates

这里的目标字段是 `label_id`。原 Pahang CSV 中名为 `class_id` 的字段实际记录空间组号（1–46），不能作为七类标签使用。

In [ ]:
RANDOM_SEED = 42
N_SPLITS = 5
INNER_SPLITS = 3
MAX_PIXELS_PER_SAMPLE = 100
SVC_CACHE_MB = 1200
WORKFLOW_VERSION = 'pahang_s2_svm_grouped_v1_20260712'

CLASS_TO_ID = {
    'Built-up/Bare soil': 0,
    'Durian': 1,
    'Forest': 2,
    'Mixed agriculture': 3,
    'Oil palm': 4,
    'Rubber': 5,
    'Water': 6,
}
ID_TO_CLASS = {class_id: name for name, class_id in CLASS_TO_ID.items()}
CLASS_IDS = sorted(ID_TO_CLASS)
CLASS_NAMES = [ID_TO_CLASS[class_id] for class_id in CLASS_IDS]
DURIAN_ID = CLASS_TO_ID['Durian']
RUBBER_ID = CLASS_TO_ID['Rubber']

S2_FEATURES = [
    'B3', 'B4', 'B5', 'B6', 'B7',
    'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDRE', 'NDWI', 'EVI',
]

LINEAR_SCREENING_PARAMS = {
    'candidate_id': 'SCREEN_LINEAR',
    'model_type': 'linear_svc',
    'C': 1.0,
    'tol': 1e-4,
    'max_iter': 20_000,
}

SVM_CANDIDATES = [
    {
        'candidate_id': 'L0', 'model_type': 'linear_svc',
        'C': 1.0, 'tol': 1e-4, 'max_iter': 20_000,
    },
    {
        'candidate_id': 'R0', 'model_type': 'rbf_svc',
        'C': 1.0, 'gamma': 'scale', 'tol': 1e-3,
    },
    {
        'candidate_id': 'R1', 'model_type': 'rbf_svc',
        'C': 10.0, 'gamma': 'scale', 'tol': 1e-3,
    },
    {
        'candidate_id': 'R2', 'model_type': 'rbf_svc',
        'C': 10.0, 'gamma': 0.01, 'tol': 1e-3,
    },
    {
        'candidate_id': 'R3', 'model_type': 'rbf_svc',
        'C': 10.0, 'gamma': 0.1, 'tol': 1e-3,
    },
]
CANDIDATE_BY_ID = {
    item['candidate_id']: item for item in SVM_CANDIDATES
}

# Bentong Rubber totals. In each Pahang outer-training split, use about 4/5
# of these totals so that reduced training mirrors a Bentong 5-fold split.
BENTONG_RUBBER_GROUPS_TOTAL = 10
BENTONG_RUBBER_POLYGONS_TOTAL = 36
TARGET_RUBBER_GROUPS_TRAIN = int(round(
    BENTONG_RUBBER_GROUPS_TOTAL * (N_SPLITS - 1) / N_SPLITS
))
TARGET_RUBBER_POLYGONS_TRAIN = int(round(
    BENTONG_RUBBER_POLYGONS_TOTAL * (N_SPLITS - 1) / N_SPLITS
))

print('Class mapping:', CLASS_TO_ID)
print('S2 predictor count:', len(S2_FEATURES))
print('Outer / inner folds:', N_SPLITS, '/', INNER_SPLITS)
print('SVM candidates:')
display(pd.DataFrame(SVM_CANDIDATES))
print(
    'Reduced Rubber target per outer training split:',
    TARGET_RUBBER_GROUPS_TRAIN, 'groups /',
    TARGET_RUBBER_POLYGONS_TRAIN, 'polygons',
)

## 1. 可复现、原子写入与 checkpoint 工具

In [ ]:
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Not JSON serializable: {type(value)}')


def atomic_write_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as file:
        json.dump(
            data, file, indent=2, ensure_ascii=False,
            default=json_default,
        )
    os.replace(temporary, path)


def atomic_write_csv(frame, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=index)
    os.replace(temporary, path)


def atomic_joblib_dump(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    joblib.dump(data, temporary, compress=3)
    os.replace(temporary, path)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def stable_seed(text, base_seed=RANDOM_SEED):
    digest = hashlib.sha256(str(text).encode('utf-8')).hexdigest()
    return (int(digest[:8], 16) + int(base_seed)) % (2**32 - 1)


INPUT_SHA256 = sha256_file(INPUT_CSV)
run_signature = {
    'workflow_version': WORKFLOW_VERSION,
    'input_csv_sha256': INPUT_SHA256,
    'random_seed': RANDOM_SEED,
    'n_splits': N_SPLITS,
    'inner_splits': INNER_SPLITS,
    'max_pixels_per_sample': MAX_PIXELS_PER_SAMPLE,
    'features': S2_FEATURES,
    'class_to_id': CLASS_TO_ID,
    'candidates': SVM_CANDIDATES,
}
signature_path = METADATA_DIR / 'run_signature.json'
if signature_path.exists():
    old_signature = load_json(signature_path)
    if old_signature != run_signature:
        raise ValueError(
            'Existing output directory has an incompatible run signature. '
            'Use a new RUN_NAME instead of mixing checkpoints.'
        )
else:
    atomic_write_json(run_signature, signature_path)

print('Input SHA256:', INPUT_SHA256)
print('Run signature locked.')

## 2. 读取并审计 Pahang CSV

本节检查字段、标签、`pixel_uid` 唯一性、每个 polygon 的类别/空间组一致性，以及 predictor 缺失值。随后按 `sample_uid` 稳定抽样，每个 polygon 最多保留100个像元。

In [ ]:
REQUIRED_COLUMNS = [
    'pixel_uid', 'sample_uid', 'group_uid',
    'class_lv2', 'label_id',
] + S2_FEATURES

header_columns = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(header_columns))
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

optional_columns = [
    column for column in [
        'spatial_group_no', 'domain', 'longitude',
        'latitude', 'valid_count',
    ]
    if column in header_columns
]

raw_df = pd.read_csv(
    INPUT_CSV,
    usecols=REQUIRED_COLUMNS + optional_columns,
    low_memory=False,
)
print(f'Raw rows: {len(raw_df):,}')
print(f'Raw memory: {raw_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

for column in ['pixel_uid', 'sample_uid', 'group_uid', 'class_lv2']:
    raw_df[column] = raw_df[column].astype('string')

raw_df['label_id'] = pd.to_numeric(
    raw_df['label_id'], errors='coerce'
)
for feature in S2_FEATURES:
    raw_df[feature] = pd.to_numeric(
        raw_df[feature], errors='coerce'
    ).astype('float32')

raw_df = raw_df.replace([np.inf, -np.inf], np.nan)
invalid_mask = raw_df[
    ['pixel_uid', 'sample_uid', 'group_uid', 'class_lv2', 'label_id']
    + S2_FEATURES
].isna().any(axis=1)

if invalid_mask.any():
    invalid_rows = raw_df.loc[
        invalid_mask,
        ['pixel_uid', 'sample_uid', 'group_uid', 'class_lv2', 'label_id'],
    ]
    atomic_write_csv(
        invalid_rows,
        TABLE_DIR / 'dropped_rows_missing_required_values.csv',
    )
    print('Dropped invalid rows:', f'{invalid_mask.sum():,}')

df = raw_df.loc[~invalid_mask].copy()
del raw_df
gc.collect()

df['label_id'] = df['label_id'].astype('int16')
# The modelling target is label_id. The source CSV's class_id was a spatial
# group number, so create a clean seven-class class_id here.
df['class_id'] = df['label_id'].astype('int16')

if df['pixel_uid'].duplicated().any():
    raise ValueError('pixel_uid must be globally unique.')

unknown_ids = sorted(set(df['class_id'].astype(int)) - set(CLASS_IDS))
if unknown_ids:
    raise ValueError(f'Unknown label_id values: {unknown_ids}')

expected_ids = df['class_lv2'].map(CLASS_TO_ID)
mismatch_mask = expected_ids.astype('int16') != df['class_id']
if mismatch_mask.any():
    mismatches = df.loc[
        mismatch_mask,
        ['sample_uid', 'class_lv2', 'label_id'],
    ].drop_duplicates()
    atomic_write_csv(mismatches, TABLE_DIR / 'class_label_mismatches.csv')
    raise ValueError(
        f'class_lv2 and label_id mismatch in {mismatch_mask.sum():,} rows.'
    )

sample_consistency = (
    df.groupby('sample_uid')
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        class_id_nunique=('class_id', 'nunique'),
        class_lv2_nunique=('class_lv2', 'nunique'),
        group_uid_nunique=('group_uid', 'nunique'),
        class_id=('class_id', 'first'),
        class_lv2=('class_lv2', 'first'),
        group_uid=('group_uid', 'first'),
    )
    .reset_index()
)
bad_samples = sample_consistency.loc[
    (sample_consistency['class_id_nunique'] != 1)
    | (sample_consistency['class_lv2_nunique'] != 1)
    | (sample_consistency['group_uid_nunique'] != 1)
]
if len(bad_samples):
    atomic_write_csv(bad_samples, TABLE_DIR / 'bad_sample_consistency.csv')
    raise ValueError(f'Inconsistent sample_uid records: {len(bad_samples)}')

group_consistency = (
    sample_consistency.groupby('group_uid')
    .agg(
        polygon_count=('sample_uid', 'size'),
        class_id_nunique=('class_id', 'nunique'),
        class_lv2_nunique=('class_lv2', 'nunique'),
        class_id=('class_id', 'first'),
        class_lv2=('class_lv2', 'first'),
    )
    .reset_index()
)
bad_groups = group_consistency.loc[
    (group_consistency['class_id_nunique'] != 1)
    | (group_consistency['class_lv2_nunique'] != 1)
]
if len(bad_groups):
    atomic_write_csv(bad_groups, TABLE_DIR / 'bad_group_consistency.csv')
    raise ValueError(f'group_uid crosses classes: {len(bad_groups)}')

raw_class_summary = (
    df.groupby(['class_id', 'class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        polygon_count=('sample_uid', 'nunique'),
        group_count=('group_uid', 'nunique'),
    )
    .reset_index()
    .sort_values('class_id')
)
atomic_write_csv(raw_class_summary, TABLE_DIR / 'pahang_raw_class_summary.csv')
atomic_write_csv(group_consistency, TABLE_DIR / 'pahang_group_summary.csv')
display(raw_class_summary)
print('Raw data QA passed.')

In [ ]:
retained_parts = []
for sample_uid, group in df.groupby('sample_uid', sort=True):
    group = group.sort_values('pixel_uid')
    if len(group) > MAX_PIXELS_PER_SAMPLE:
        group = group.sample(
            n=MAX_PIXELS_PER_SAMPLE,
            random_state=stable_seed(sample_uid),
            replace=False,
        )
    retained_parts.append(group)

model_df = (
    pd.concat(retained_parts, ignore_index=True)
    .sort_values('pixel_uid')
    .reset_index(drop=True)
)
del retained_parts, df
gc.collect()

retained_counts = model_df.groupby('sample_uid')['pixel_uid'].transform('size')
model_df['sample_weight'] = (1.0 / retained_counts).astype('float64')

polygon_weight_sums = model_df.groupby('sample_uid')['sample_weight'].sum()
if not np.allclose(polygon_weight_sums.to_numpy(), 1.0):
    raise AssertionError('Each polygon must have total sample_weight = 1.')

model_class_summary = (
    model_df.groupby(['class_id', 'class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        polygon_count=('sample_uid', 'nunique'),
        group_count=('group_uid', 'nunique'),
        polygon_weight_sum=('sample_weight', 'sum'),
    )
    .reset_index()
    .sort_values('class_id')
)
atomic_write_csv(
    model_class_summary,
    TABLE_DIR / 'pahang_model_class_summary.csv',
)

print(f'Retained model rows: {len(model_df):,}')
print('Polygons:', model_df['sample_uid'].nunique())
print('Spatial groups:', model_df['group_uid'].nunique())
print('Maximum retained pixels per polygon:', MAX_PIXELS_PER_SAMPLE)
display(model_class_summary)

## 3. 建立5折 outer spatial grouped assignments

fold 是在 polygon 表上建立的；同一 `group_uid` 必须完整进入同一个 fold。函数会尝试多个随机种子，只接受每个验证 fold 都包含七类的方案，并同时平衡 polygon 数与空间组数。

In [ ]:
def build_valid_grouped_assignments(
    samples,
    n_splits,
    base_seed,
    fold_column,
    max_attempts=1000,
):
    samples = samples.reset_index(drop=True).copy()
    dummy_x = np.zeros((len(samples), 1))
    best_fold_ids = None
    best_seed = None
    best_score = np.inf

    for attempt in range(max_attempts):
        seed = base_seed + attempt
        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed,
        )
        fold_ids = np.full(len(samples), -1, dtype=int)
        for fold_id, (_, validation_indices) in enumerate(
            splitter.split(
                dummy_x,
                samples['class_id'],
                groups=samples['group_uid'],
            )
        ):
            fold_ids[validation_indices] = fold_id

        test_frame = samples.assign(_fold=fold_ids)
        class_counts = pd.crosstab(
            test_frame['_fold'], test_frame['class_id']
        ).reindex(
            index=range(n_splits),
            columns=CLASS_IDS,
            fill_value=0,
        )
        if (class_counts == 0).any().any():
            continue

        group_counts = (
            test_frame.groupby(['_fold', 'class_id'])['group_uid']
            .nunique()
            .unstack(fill_value=0)
            .reindex(
                index=range(n_splits),
                columns=CLASS_IDS,
                fill_value=0,
            )
        )
        expected = 1.0 / n_splits
        class_proportions = class_counts.div(
            class_counts.sum(axis=0), axis=1
        )
        group_proportions = group_counts.div(
            group_counts.sum(axis=0), axis=1
        )
        score = (
            ((class_proportions - expected) ** 2).to_numpy().sum()
            + ((group_proportions - expected) ** 2).to_numpy().sum()
        )
        if score < best_score:
            best_fold_ids = fold_ids.copy()
            best_seed = seed
            best_score = score

    if best_fold_ids is None:
        raise ValueError('Could not create grouped folds containing all classes.')

    assignments = samples[
        ['sample_uid', 'group_uid', 'class_id', 'class_lv2']
    ].copy()
    assignments[fold_column] = best_fold_ids
    assignments['split_seed'] = best_seed
    assignments['balance_score'] = best_score

    if assignments.groupby('group_uid')[fold_column].nunique().max() != 1:
        raise AssertionError('A spatial group crosses folds.')
    return assignments


sample_table = (
    model_df[
        ['sample_uid', 'group_uid', 'class_id', 'class_lv2']
    ]
    .drop_duplicates('sample_uid')
    .sort_values('sample_uid')
    .reset_index(drop=True)
)

outer_assignment_path = TABLE_DIR / 'fold_assignments.csv'
if outer_assignment_path.exists():
    outer_assignments = pd.read_csv(
        outer_assignment_path,
        dtype={
            'sample_uid': 'string',
            'group_uid': 'string',
            'class_lv2': 'string',
        },
    )
    if set(outer_assignments['sample_uid'].astype(str)) != set(
        sample_table['sample_uid'].astype(str)
    ):
        raise ValueError('Existing outer fold assignments are stale.')
    print('Reused existing outer fold assignments.')
else:
    outer_assignments = build_valid_grouped_assignments(
        sample_table,
        n_splits=N_SPLITS,
        base_seed=RANDOM_SEED,
        fold_column='fold_id',
    )
    atomic_write_csv(outer_assignments, outer_assignment_path)
    print('Created outer fold assignments.')

sample_table = sample_table.merge(
    outer_assignments[['sample_uid', 'fold_id']],
    on='sample_uid',
    how='left',
    validate='one_to_one',
)
model_df = model_df.merge(
    outer_assignments[['sample_uid', 'fold_id']],
    on='sample_uid',
    how='left',
    validate='many_to_one',
)
model_df['fold_id'] = model_df['fold_id'].astype('int8')

fold_class_counts = pd.crosstab(
    sample_table['fold_id'], sample_table['class_id']
).reindex(index=range(N_SPLITS), columns=CLASS_IDS, fill_value=0)
if (fold_class_counts == 0).any().any():
    raise ValueError('At least one outer fold is missing a class.')
if sample_table.groupby('group_uid')['fold_id'].nunique().max() != 1:
    raise AssertionError('Outer spatial leakage detected.')

training_row_columns = [
    'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2',
    'class_id', 'fold_id', 'sample_weight',
]
atomic_write_csv(
    model_df[training_row_columns],
    TABLE_DIR / 'training_rows_used.csv',
)
atomic_write_csv(fold_class_counts, TABLE_DIR / 'outer_fold_class_counts.csv', index=True)

display(fold_class_counts.rename(columns=ID_TO_CLASS))
print('Outer grouped fold QA passed.')

## 4. SVM、权重、预测聚合与指标函数

### 4.1 无泄漏 StandardScaler–SVM pipeline

与 Bentong 相同：scaler 只在当前训练 split 中拟合；每个 polygon 总权重相等，同时对训练 split 的类别频率进行平衡。

In [ ]:
def make_svm_training_weights(train_frame):
    class_row_counts = train_frame['class_id'].value_counts()
    if set(class_row_counts.index.astype(int)) != set(CLASS_IDS):
        raise ValueError('A training split is missing a class.')
    class_factors = {
        int(class_id): len(train_frame)
        / (len(CLASS_IDS) * int(row_count))
        for class_id, row_count in class_row_counts.items()
    }
    weights = (
        train_frame['sample_weight'].to_numpy(dtype='float64')
        * train_frame['class_id'].map(class_factors).to_numpy(dtype='float64')
    )
    weights *= len(weights) / weights.sum()
    return weights.astype('float64')


def build_svm_pipeline(params):
    if params['model_type'] == 'linear_svc':
        classifier = LinearSVC(
            C=float(params['C']),
            tol=float(params.get('tol', 1e-4)),
            max_iter=int(params.get('max_iter', 20_000)),
            dual='auto',
            class_weight=None,
            random_state=RANDOM_SEED,
        )
    elif params['model_type'] == 'rbf_svc':
        classifier = SVC(
            C=float(params['C']),
            kernel='rbf',
            gamma=params['gamma'],
            tol=float(params.get('tol', 1e-3)),
            probability=False,
            class_weight=None,
            cache_size=SVC_CACHE_MB,
            shrinking=True,
            decision_function_shape='ovr',
            break_ties=True,
            max_iter=-1,
            random_state=RANDOM_SEED,
        )
    else:
        raise ValueError(f'Unknown model_type: {params["model_type"]}')

    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', classifier),
    ])


def fit_svm(train_frame, features, params):
    model = build_svm_pipeline(params)
    model.fit(
        train_frame[features],
        train_frame['class_id'],
        scaler__sample_weight=train_frame[
            'sample_weight'
        ].to_numpy(dtype='float64'),
        classifier__sample_weight=make_svm_training_weights(train_frame),
    )
    classifier = model.named_steps['classifier']
    if isinstance(classifier, LinearSVC):
        observed_iterations = int(np.max(np.atleast_1d(classifier.n_iter_)))
        if observed_iterations >= int(params.get('max_iter', 20_000)):
            warnings.warn(
                'LinearSVC reached max_iter; review convergence.',
                RuntimeWarning,
            )
    return model

### 4.2 像元 decision scores、polygon 聚合和评价指标

In [ ]:
def make_pixel_predictions(model, frame, features):
    predicted_ids = model.predict(frame[features]).astype('int16')
    decision_scores = model.decision_function(frame[features])
    model_classes = np.asarray(
        model.named_steps['classifier'].classes_
    ).astype(int)

    if decision_scores.ndim != 2 or decision_scores.shape[1] != len(model_classes):
        raise ValueError('Unexpected multiclass decision score shape.')

    output = frame[[
        'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2',
        'class_id', 'fold_id', 'sample_weight',
    ]].copy()
    output['pred_class_id'] = predicted_ids
    output['pred_class_name'] = output['pred_class_id'].map(ID_TO_CLASS)

    for class_id in CLASS_IDS:
        output[f'score_{class_id}'] = np.nan
    for column_index, class_id in enumerate(model_classes):
        output[f'score_{class_id}'] = decision_scores[:, column_index]
    score_columns = [f'score_{class_id}' for class_id in CLASS_IDS]
    if output[score_columns].isna().any().any():
        raise ValueError('At least one class decision score is missing.')
    return output


def aggregate_polygon_predictions(pixel_predictions):
    score_columns = [f'score_{class_id}' for class_id in CLASS_IDS]
    first_part = (
        pixel_predictions[[
            'sample_uid', 'group_uid', 'class_lv2',
            'class_id', 'fold_id',
        ]]
        .groupby('sample_uid', as_index=False)
        .first()
    )
    score_part = (
        pixel_predictions[['sample_uid'] + score_columns]
        .groupby('sample_uid', as_index=False)
        .mean()
    )
    polygon_predictions = first_part.merge(
        score_part,
        on='sample_uid',
        how='left',
        validate='one_to_one',
    )
    score_matrix = polygon_predictions[score_columns].to_numpy()
    polygon_predictions['pred_class_id'] = np.asarray(CLASS_IDS)[
        score_matrix.argmax(axis=1)
    ].astype('int16')
    polygon_predictions['pred_class_name'] = polygon_predictions[
        'pred_class_id'
    ].map(ID_TO_CLASS)
    return polygon_predictions


def one_class_prf(y_true, y_pred, class_id, sample_weight=None):
    precision, recall, score, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[class_id],
        average=None,
        sample_weight=sample_weight,
        zero_division=0,
    )
    return float(precision[0]), float(recall[0]), float(score[0])


def metric_dictionary(y_true, y_pred, sample_weight=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    durian_precision, durian_recall, durian_f1 = one_class_prf(
        y_true, y_pred, DURIAN_ID, sample_weight
    )
    rubber_precision, rubber_recall, rubber_f1 = one_class_prf(
        y_true, y_pred, RUBBER_ID, sample_weight
    )
    return {
        'accuracy': float(accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        )),
        'balanced_accuracy': float(balanced_accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        )),
        'macro_f1': float(f1_score(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            average='macro',
            sample_weight=sample_weight,
            zero_division=0,
        )),
        'durian_precision': durian_precision,
        'durian_recall': durian_recall,
        'durian_f1': durian_f1,
        'rubber_precision': rubber_precision,
        'rubber_recall': rubber_recall,
        'rubber_f1': rubber_f1,
    }


def evaluate_model(model, validation_frame, features):
    pixel_predictions = make_pixel_predictions(
        model, validation_frame, features
    )
    polygon_predictions = aggregate_polygon_predictions(pixel_predictions)

    pixel_metrics = metric_dictionary(
        pixel_predictions['class_id'],
        pixel_predictions['pred_class_id'],
        sample_weight=pixel_predictions['sample_weight'],
    )
    polygon_metrics = metric_dictionary(
        polygon_predictions['class_id'],
        polygon_predictions['pred_class_id'],
    )
    metrics = {
        **{f'pixel_{key}': value for key, value in pixel_metrics.items()},
        **{f'polygon_{key}': value for key, value in polygon_metrics.items()},
    }
    return metrics, pixel_predictions, polygon_predictions


def report_frame(y_true, y_pred, sample_weight=None):
    return pd.DataFrame(classification_report(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        target_names=CLASS_NAMES,
        sample_weight=sample_weight,
        output_dict=True,
        zero_division=0,
    )).T


def save_confusion_outputs(
    y_true,
    y_pred,
    title,
    stem,
    sample_weight=None,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        sample_weight=sample_weight,
    )
    frame = pd.DataFrame(
        matrix,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )
    atomic_write_csv(frame, TABLE_DIR / f'{stem}.csv', index=True)
    plt.figure(figsize=(9, 7))
    sns.heatmap(frame, annot=True, fmt='.1f', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted class')
    plt.ylabel('True class')
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / f'{stem}.png', dpi=180, bbox_inches='tight'
    )
    plt.show()
    return frame

### 4.3 Inner-fold assignments 与逐 fold checkpoint 评分

In [ ]:
SELECTION_METRICS = [
    'polygon_durian_f1_mean',
    'polygon_macro_f1_mean',
    'pixel_durian_f1_mean',
]


def get_or_create_inner_assignments(outer_fold_id, outer_train_samples):
    path = (
        CHECKPOINT_DIR / 'nested' / f'outer_{outer_fold_id}'
        / 'inner_fold_assignments.csv'
    )
    if path.exists():
        assignments = pd.read_csv(
            path,
            dtype={
                'sample_uid': 'string',
                'group_uid': 'string',
                'class_lv2': 'string',
            },
        )
        if set(assignments['sample_uid'].astype(str)) != set(
            outer_train_samples['sample_uid'].astype(str)
        ):
            raise ValueError('Stale inner fold assignments.')
        return assignments

    assignments = build_valid_grouped_assignments(
        outer_train_samples,
        n_splits=INNER_SPLITS,
        base_seed=RANDOM_SEED + 2000 + outer_fold_id * 1000,
        fold_column='inner_fold_id',
    )
    atomic_write_csv(assignments, path)
    print(
        f'Outer {outer_fold_id}: inner seed=',
        int(assignments['split_seed'].iloc[0]),
    )
    return assignments


def score_configuration_checkpointed(
    outer_train_pixels,
    inner_assignments,
    features,
    params,
    checkpoint_directory,
):
    checkpoint_directory = Path(checkpoint_directory)
    checkpoint_directory.mkdir(parents=True, exist_ok=True)
    fold_records = []

    for inner_fold_id in range(INNER_SPLITS):
        checkpoint_path = (
            checkpoint_directory / f'inner_{inner_fold_id}.json'
        )
        if checkpoint_path.exists():
            record = load_json(checkpoint_path)
            print(f'    inner {inner_fold_id}: reused')
            fold_records.append(record)
            continue

        validation_uids = set(
            inner_assignments.loc[
                inner_assignments['inner_fold_id'] == inner_fold_id,
                'sample_uid',
            ].astype(str)
        )
        train_frame = outer_train_pixels.loc[
            ~outer_train_pixels['sample_uid'].astype(str).isin(validation_uids)
        ]
        validation_frame = outer_train_pixels.loc[
            outer_train_pixels['sample_uid'].astype(str).isin(validation_uids)
        ]
        if set(train_frame['group_uid']) & set(validation_frame['group_uid']):
            raise AssertionError('Inner group leakage.')

        fit_start = time.time()
        model = fit_svm(train_frame, features, params)
        metrics, pixel_predictions, polygon_predictions = evaluate_model(
            model, validation_frame, features
        )
        record = {
            'inner_fold_id': inner_fold_id,
            'runtime_minutes': (time.time() - fit_start) / 60,
            **metrics,
        }
        atomic_write_json(record, checkpoint_path)
        fold_records.append(record)
        del model, pixel_predictions, polygon_predictions
        gc.collect()
        print(
            f'    inner {inner_fold_id}: completed in '
            f'{record["runtime_minutes"]:.1f} min'
        )

    fold_frame = pd.DataFrame(fold_records)
    return {
        'polygon_durian_f1_mean': float(
            fold_frame['polygon_durian_f1'].mean()
        ),
        'polygon_macro_f1_mean': float(
            fold_frame['polygon_macro_f1'].mean()
        ),
        'pixel_durian_f1_mean': float(
            fold_frame['pixel_durian_f1'].mean()
        ),
        'pixel_macro_f1_mean': float(
            fold_frame['pixel_macro_f1'].mean()
        ),
        'polygon_rubber_f1_mean': float(
            fold_frame['polygon_rubber_f1'].mean()
        ),
        'pixel_rubber_f1_mean': float(
            fold_frame['pixel_rubber_f1'].mean()
        ),
        'runtime_minutes_sum': float(
            fold_frame['runtime_minutes'].sum()
        ),
    }


def rank_candidate_frame(frame):
    return frame.sort_values(
        SELECTION_METRICS + ['candidate_id'],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

## 5. 正式 nested grouped validation

对每个 outer training set：

1. 建立3折 inner spatial groups；
2. 用相同 S2 predictors 比较5个 SVM candidates；
3. 按 Bentong 的顺序选择：polygon Durian F1 → polygon macro F1 → pixel Durian F1；
4. 在完整 outer training set 上训练所选 candidate，并预测未见过的 outer validation groups。

这部分输出的 OOF 指标是 Pahang 内部验证的正式无空间组泄漏结果。

In [ ]:
nested_candidate_records = []
nested_outer_records = []
nested_selected_candidates = []

for outer_fold_id in range(N_SPLITS):
    print(f'\n===== OUTER FOLD {outer_fold_id} =====')
    outer_train_pixels = model_df.loc[
        model_df['fold_id'] != outer_fold_id
    ].copy()
    outer_validation_pixels = model_df.loc[
        model_df['fold_id'] == outer_fold_id
    ].copy()
    if set(outer_train_pixels['group_uid']) & set(
        outer_validation_pixels['group_uid']
    ):
        raise AssertionError('Outer group leakage.')

    outer_train_samples = sample_table.loc[
        sample_table['fold_id'] != outer_fold_id,
        ['sample_uid', 'group_uid', 'class_id', 'class_lv2'],
    ].copy()
    inner_assignments = get_or_create_inner_assignments(
        outer_fold_id, outer_train_samples
    )

    outer_candidate_records = []
    for params in SVM_CANDIDATES:
        candidate_id = params['candidate_id']
        print('  Candidate:', candidate_id)
        scores = score_configuration_checkpointed(
            outer_train_pixels=outer_train_pixels,
            inner_assignments=inner_assignments,
            features=S2_FEATURES,
            params=params,
            checkpoint_directory=(
                CHECKPOINT_DIR / 'nested' / f'outer_{outer_fold_id}'
                / 'candidates' / candidate_id
            ),
        )
        record = {
            'outer_fold_id': outer_fold_id,
            'candidate_id': candidate_id,
            **params,
            **scores,
        }
        outer_candidate_records.append(record)
        nested_candidate_records.append(record)

    ranked = rank_candidate_frame(pd.DataFrame(outer_candidate_records))
    selected_record = ranked.iloc[0].to_dict()
    selected_candidate_id = selected_record['candidate_id']
    selected_params = CANDIDATE_BY_ID[selected_candidate_id]
    nested_selected_candidates.append({
        'outer_fold_id': outer_fold_id,
        'selected_candidate_id': selected_candidate_id,
        **{
            metric: selected_record[metric]
            for metric in SELECTION_METRICS
        },
    })
    print('  Selected candidate:', selected_candidate_id)

    outer_dir = (
        CHECKPOINT_DIR / 'nested' / f'outer_{outer_fold_id}'
        / 'outer_evaluation'
    )
    outer_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = outer_dir / 'metrics.json'
    pixel_path = outer_dir / 'pixel_predictions.csv'
    polygon_path = outer_dir / 'polygon_predictions.csv'

    if metrics_path.exists() and pixel_path.exists() and polygon_path.exists():
        outer_metrics = load_json(metrics_path)
        print('  Outer predictions reused.')
    else:
        outer_start = time.time()
        selected_model = fit_svm(
            outer_train_pixels, S2_FEATURES, selected_params
        )
        metrics, pixel_predictions, polygon_predictions = evaluate_model(
            selected_model, outer_validation_pixels, S2_FEATURES
        )
        outer_metrics = {
            'outer_fold_id': outer_fold_id,
            'selected_candidate_id': selected_candidate_id,
            'runtime_minutes': (time.time() - outer_start) / 60,
            **metrics,
        }
        atomic_write_csv(pixel_predictions, pixel_path)
        atomic_write_csv(polygon_predictions, polygon_path)
        atomic_write_json(outer_metrics, metrics_path)
        del selected_model, pixel_predictions, polygon_predictions
        gc.collect()
        print(
            '  Outer evaluation completed in',
            f'{outer_metrics["runtime_minutes"]:.1f} min',
        )

    nested_outer_records.append(outer_metrics)

nested_candidate_scores = pd.DataFrame(nested_candidate_records)
nested_fold_metrics = pd.DataFrame(nested_outer_records)
selected_candidates_by_fold = pd.DataFrame(nested_selected_candidates)

atomic_write_csv(
    nested_candidate_scores,
    TABLE_DIR / 'nested_inner_candidate_scores.csv',
)
atomic_write_csv(
    nested_fold_metrics,
    TABLE_DIR / 'nested_outer_fold_metrics.csv',
)
atomic_write_csv(
    selected_candidates_by_fold,
    TABLE_DIR / 'nested_selected_candidates.csv',
)

display(selected_candidates_by_fold)
display(nested_fold_metrics)

## 6. 合并 OOF 预测并生成正式 Pahang 内部验证结果

In [ ]:
oof_pixel_parts = []
oof_polygon_parts = []
for outer_fold_id in range(N_SPLITS):
    outer_dir = (
        CHECKPOINT_DIR / 'nested' / f'outer_{outer_fold_id}'
        / 'outer_evaluation'
    )
    oof_pixel_parts.append(pd.read_csv(
        outer_dir / 'pixel_predictions.csv', low_memory=False
    ))
    oof_polygon_parts.append(pd.read_csv(
        outer_dir / 'polygon_predictions.csv', low_memory=False
    ))

oof_pixel_predictions = pd.concat(oof_pixel_parts, ignore_index=True)
oof_polygon_predictions = pd.concat(oof_polygon_parts, ignore_index=True)
del oof_pixel_parts, oof_polygon_parts
gc.collect()

if oof_pixel_predictions['pixel_uid'].duplicated().any():
    raise AssertionError('Duplicate OOF pixel predictions.')
if oof_polygon_predictions['sample_uid'].duplicated().any():
    raise AssertionError('Duplicate OOF polygon predictions.')
if len(oof_pixel_predictions) != len(model_df):
    raise AssertionError('OOF pixel prediction count mismatch.')
if len(oof_polygon_predictions) != sample_table['sample_uid'].nunique():
    raise AssertionError('OOF polygon prediction count mismatch.')

atomic_write_csv(
    oof_pixel_predictions,
    TABLE_DIR / 'oof_pixel_predictions.csv',
)
atomic_write_csv(
    oof_polygon_predictions,
    TABLE_DIR / 'oof_polygon_predictions.csv',
)

pixel_metrics = metric_dictionary(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    sample_weight=oof_pixel_predictions['sample_weight'],
)
polygon_metrics = metric_dictionary(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
nested_overall_metrics = {
    **{f'pixel_{key}': value for key, value in pixel_metrics.items()},
    **{f'polygon_{key}': value for key, value in polygon_metrics.items()},
    'outer_fold_selected_candidate_counts': (
        selected_candidates_by_fold['selected_candidate_id']
        .value_counts().to_dict()
    ),
    'feature_set': 'S2',
}
atomic_write_json(
    nested_overall_metrics,
    METADATA_DIR / 'nested_overall_metrics.json',
)

pixel_report = report_frame(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    sample_weight=oof_pixel_predictions['sample_weight'],
)
polygon_report = report_frame(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
atomic_write_csv(
    pixel_report,
    TABLE_DIR / 'oof_pixel_classification_report.csv',
    index=True,
)
atomic_write_csv(
    polygon_report,
    TABLE_DIR / 'oof_polygon_classification_report.csv',
    index=True,
)

save_confusion_outputs(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    title='Pahang nested grouped OOF — pixel weighted',
    stem='oof_pixel_confusion_matrix',
    sample_weight=oof_pixel_predictions['sample_weight'],
)
save_confusion_outputs(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
    title='Pahang nested grouped OOF — polygon',
    stem='oof_polygon_confusion_matrix',
)

print(json.dumps(nested_overall_metrics, indent=2))
print('Polygon classification report:')
display(polygon_report)

## 7. 选择部署 candidate 并在全部 Pahang 数据上训练最终 SVM

正式性能仍以上一节 nested OOF 为准。本节沿用 Bentong 做法，在固定 outer folds 上比较 candidate，选择一套用于最终全数据模型的参数。

In [ ]:
final_candidate_records = []
final_checkpoint_dir = CHECKPOINT_DIR / 'final_candidate_selection'
final_checkpoint_dir.mkdir(parents=True, exist_ok=True)

for params in SVM_CANDIDATES:
    candidate_id = params['candidate_id']
    print('\nFinal candidate:', candidate_id)
    for fold_id in range(N_SPLITS):
        checkpoint_path = (
            final_checkpoint_dir / f'{candidate_id}_fold_{fold_id}.json'
        )
        if checkpoint_path.exists():
            record = load_json(checkpoint_path)
            print(f'  fold {fold_id}: reused')
        else:
            train_frame = model_df.loc[model_df['fold_id'] != fold_id]
            validation_frame = model_df.loc[model_df['fold_id'] == fold_id]
            fit_start = time.time()
            candidate_model = fit_svm(
                train_frame, S2_FEATURES, params
            )
            metrics, pixel_predictions, polygon_predictions = evaluate_model(
                candidate_model, validation_frame, S2_FEATURES
            )
            record = {
                'candidate_id': candidate_id,
                'fold_id': fold_id,
                'runtime_minutes': (time.time() - fit_start) / 60,
                **metrics,
            }
            atomic_write_json(record, checkpoint_path)
            del candidate_model, pixel_predictions, polygon_predictions
            gc.collect()
            print(
                f'  fold {fold_id}: completed in '
                f'{record["runtime_minutes"]:.1f} min'
            )
        final_candidate_records.append(record)

final_candidate_fold_metrics = pd.DataFrame(final_candidate_records)
final_candidate_scores = (
    final_candidate_fold_metrics.groupby('candidate_id')
    .agg(
        polygon_durian_f1_mean=('polygon_durian_f1', 'mean'),
        polygon_macro_f1_mean=('polygon_macro_f1', 'mean'),
        pixel_durian_f1_mean=('pixel_durian_f1', 'mean'),
        pixel_macro_f1_mean=('pixel_macro_f1', 'mean'),
        polygon_rubber_f1_mean=('polygon_rubber_f1', 'mean'),
        pixel_rubber_f1_mean=('pixel_rubber_f1', 'mean'),
        runtime_minutes_sum=('runtime_minutes', 'sum'),
    )
    .reset_index()
)
final_candidate_scores = rank_candidate_frame(final_candidate_scores)
FINAL_CANDIDATE_ID = final_candidate_scores.iloc[0]['candidate_id']
FINAL_PARAMS = CANDIDATE_BY_ID[FINAL_CANDIDATE_ID]

atomic_write_csv(
    final_candidate_fold_metrics,
    TABLE_DIR / 'final_candidate_fold_metrics.csv',
)
atomic_write_csv(
    final_candidate_scores,
    TABLE_DIR / 'final_candidate_scores.csv',
)

print('Selected final candidate:', FINAL_CANDIDATE_ID)
display(final_candidate_scores)

In [ ]:
final_model_path = MODEL_DIR / 'pahang_svm_final_model.joblib'
final_bundle_path = MODEL_DIR / 'pahang_svm_final_bundle.joblib'

if final_model_path.exists() and final_bundle_path.exists():
    final_model = joblib.load(final_model_path)
    print('Reused existing final model.')
else:
    final_start = time.time()
    final_model = fit_svm(model_df, S2_FEATURES, FINAL_PARAMS)
    atomic_joblib_dump(final_model, final_model_path)
    final_bundle = {
        'model': final_model,
        'model_type': 'SVM',
        'feature_set': 'S2',
        'predictor_bands': S2_FEATURES,
        'candidate_id': FINAL_CANDIDATE_ID,
        'svm_params': FINAL_PARAMS,
        'class_to_id': CLASS_TO_ID,
        'id_to_class': ID_TO_CLASS,
        'random_seed': RANDOM_SEED,
        'training_domain': 'Pahang excluding Bentong',
        'training_input_sha256': INPUT_SHA256,
        'sample_count': int(sample_table['sample_uid'].nunique()),
        'group_count': int(sample_table['group_uid'].nunique()),
        'pixel_count': int(len(model_df)),
        'nested_oof_metrics': nested_overall_metrics,
        'created_utc': datetime.now(timezone.utc).isoformat(),
    }
    atomic_joblib_dump(final_bundle, final_bundle_path)
    print(
        'Final full-data training minutes:',
        f'{(time.time() - final_start) / 60:.1f}',
    )

classifier = final_model.named_steps['classifier']
diagnostics = {
    'candidate_id': FINAL_CANDIDATE_ID,
    'classifier_type': type(classifier).__name__,
    'classes': [int(value) for value in classifier.classes_],
}
if hasattr(classifier, 'n_support_'):
    diagnostics['support_vectors_per_class'] = {
        ID_TO_CLASS[class_id]: int(count)
        for class_id, count in zip(CLASS_IDS, classifier.n_support_)
    }
if hasattr(classifier, 'coef_'):
    diagnostics['coefficient_shape'] = list(classifier.coef_.shape)
atomic_write_json(diagnostics, METADATA_DIR / 'final_model_diagnostics.json')
print(json.dumps(diagnostics, indent=2))

## 8. Bentong 与 Pahang 的描述性对照

如果 Drive 中保留了 Bentong RF/SVM 结果，本节自动提取两地的类别样本规模及 OOF Rubber/Durian F1。此对照只能说明现象，不能单独把差异归因于 Rubber 样本量。

In [ ]:
comparison_rows = []

pahang_counts = model_class_summary.set_index('class_lv2')
for class_name in ['Durian', 'Rubber']:
    comparison_rows.append({
        'domain': 'Pahang excluding Bentong',
        'class_lv2': class_name,
        'polygon_count': int(pahang_counts.loc[class_name, 'polygon_count']),
        'group_count': int(pahang_counts.loc[class_name, 'group_count']),
        'pixel_oof_f1': float(pixel_report.loc[class_name, 'f1-score']),
        'polygon_oof_f1': float(polygon_report.loc[class_name, 'f1-score']),
    })

bentong_training_path = BENTONG_RF_DIR / 'tables' / 'training_rows_used.csv'
bentong_pixel_report_path = (
    BENTONG_SVM_DIR / 'tables' / 'oof_pixel_classification_report.csv'
)
bentong_polygon_report_path = (
    BENTONG_SVM_DIR / 'tables' / 'oof_polygon_classification_report.csv'
)

if (
    bentong_training_path.exists()
    and bentong_pixel_report_path.exists()
    and bentong_polygon_report_path.exists()
):
    bentong_training = pd.read_csv(
        bentong_training_path,
        usecols=['sample_uid', 'group_uid', 'class_lv2'],
        low_memory=False,
    )
    bentong_counts = (
        bentong_training.groupby('class_lv2')
        .agg(
            polygon_count=('sample_uid', 'nunique'),
            group_count=('group_uid', 'nunique'),
        )
    )
    bentong_pixel_report = pd.read_csv(
        bentong_pixel_report_path, index_col=0
    )
    bentong_polygon_report = pd.read_csv(
        bentong_polygon_report_path, index_col=0
    )
    for class_name in ['Durian', 'Rubber']:
        comparison_rows.append({
            'domain': 'Bentong',
            'class_lv2': class_name,
            'polygon_count': int(
                bentong_counts.loc[class_name, 'polygon_count']
            ),
            'group_count': int(
                bentong_counts.loc[class_name, 'group_count']
            ),
            'pixel_oof_f1': float(
                bentong_pixel_report.loc[class_name, 'f1-score']
            ),
            'polygon_oof_f1': float(
                bentong_polygon_report.loc[class_name, 'f1-score']
            ),
        })
    print('Bentong comparison loaded.')
else:
    print('Bentong files not found; Pahang-only comparison table created.')

domain_comparison = pd.DataFrame(comparison_rows)
atomic_write_csv(
    domain_comparison,
    TABLE_DIR / 'bentong_vs_pahang_durian_rubber_comparison.csv',
)
display(domain_comparison)

## 9. Rubber 样本规模配对实验

这一节直接检验你的猜想：

- 每个 outer validation fold 保持完全不变；
- 模型参数、S2 predictors、权重和其他六类训练样本保持不变；
- 只把训练侧 Rubber 缩减到约8个空间组、29个 polygon（对应 Bentong 总量10组/36 polygon 在5折训练中的约4/5）；
- 每个 fold 重复5次不同的空间组抽样；
- 比较完整训练与缩减训练在相同验证样本上的 Rubber F1，以及 Rubber→Durian、Durian→Rubber 错分率。

如果缩减 Rubber 后，多数重复的 Rubber F1 明显下降，并且相关错分率上升，就支持“训练 Rubber 的空间多样性不足是性能下降的重要原因”。如果差异很小，则应优先检查域偏移、标签边界或光谱混淆，而不是只增加数量。

In [ ]:
def directed_error_rate(y_true, y_pred, true_id, predicted_id):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    mask = y_true == true_id
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(y_pred[mask] == predicted_id))


def select_reduced_rubber_training(train_frame, seed):
    rubber_samples = (
        train_frame.loc[
            train_frame['class_id'] == RUBBER_ID,
            ['sample_uid', 'group_uid'],
        ]
        .drop_duplicates('sample_uid')
        .sort_values(['group_uid', 'sample_uid'])
        .reset_index(drop=True)
    )
    available_groups = rubber_samples['group_uid'].drop_duplicates().to_numpy()
    if len(available_groups) < TARGET_RUBBER_GROUPS_TRAIN:
        raise ValueError('Not enough Rubber groups for the reduced experiment.')

    rng = np.random.default_rng(seed)
    chosen_groups = rng.choice(
        available_groups,
        size=TARGET_RUBBER_GROUPS_TRAIN,
        replace=False,
    )
    chosen_pool = rubber_samples.loc[
        rubber_samples['group_uid'].isin(chosen_groups)
    ].copy()

    # First keep one polygon from each selected group, then fill the remaining
    # polygon quota from the selected groups. This preserves the target diversity.
    mandatory_uids = []
    for group_uid in sorted(chosen_groups):
        candidates = chosen_pool.loc[
            chosen_pool['group_uid'] == group_uid,
            'sample_uid',
        ].to_numpy()
        mandatory_uids.append(rng.choice(candidates))

    mandatory_uids = list(dict.fromkeys(mandatory_uids))
    remaining_uids = chosen_pool.loc[
        ~chosen_pool['sample_uid'].isin(mandatory_uids),
        'sample_uid',
    ].to_numpy()
    extra_count = min(
        max(TARGET_RUBBER_POLYGONS_TRAIN - len(mandatory_uids), 0),
        len(remaining_uids),
    )
    if extra_count:
        extra_uids = rng.choice(
            remaining_uids, size=extra_count, replace=False
        ).tolist()
    else:
        extra_uids = []

    selected_rubber_uids = set(mandatory_uids + extra_uids)
    reduced = train_frame.loc[
        (train_frame['class_id'] != RUBBER_ID)
        | train_frame['sample_uid'].isin(selected_rubber_uids)
    ].copy()

    selected_rubber = reduced.loc[reduced['class_id'] == RUBBER_ID]
    audit = {
        'selected_rubber_polygons': int(
            selected_rubber['sample_uid'].nunique()
        ),
        'selected_rubber_groups': int(
            selected_rubber['group_uid'].nunique()
        ),
        'selected_rubber_pixels': int(len(selected_rubber)),
    }
    return reduced, audit


def ablation_metrics_record(
    condition,
    outer_fold_id,
    repeat_id,
    metrics,
    pixel_predictions,
    polygon_predictions,
    training_audit,
):
    return {
        'condition': condition,
        'outer_fold_id': outer_fold_id,
        'repeat_id': repeat_id,
        **training_audit,
        **metrics,
        'polygon_rubber_to_durian_rate': directed_error_rate(
            polygon_predictions['class_id'],
            polygon_predictions['pred_class_id'],
            RUBBER_ID,
            DURIAN_ID,
        ),
        'polygon_durian_to_rubber_rate': directed_error_rate(
            polygon_predictions['class_id'],
            polygon_predictions['pred_class_id'],
            DURIAN_ID,
            RUBBER_ID,
        ),
        'pixel_rubber_to_durian_rate': directed_error_rate(
            pixel_predictions['class_id'],
            pixel_predictions['pred_class_id'],
            RUBBER_ID,
            DURIAN_ID,
        ),
        'pixel_durian_to_rubber_rate': directed_error_rate(
            pixel_predictions['class_id'],
            pixel_predictions['pred_class_id'],
            DURIAN_ID,
            RUBBER_ID,
        ),
    }

In [ ]:
if RUN_RUBBER_ABLATION:
    ablation_dir = CHECKPOINT_DIR / 'rubber_ablation'
    ablation_dir.mkdir(parents=True, exist_ok=True)
    ablation_records = []

    for outer_fold_id in range(N_SPLITS):
        print(f'\nRubber ablation — outer fold {outer_fold_id}')
        train_frame = model_df.loc[model_df['fold_id'] != outer_fold_id]
        validation_frame = model_df.loc[model_df['fold_id'] == outer_fold_id]

        full_rubber = train_frame.loc[train_frame['class_id'] == RUBBER_ID]
        full_audit = {
            'selected_rubber_polygons': int(
                full_rubber['sample_uid'].nunique()
            ),
            'selected_rubber_groups': int(
                full_rubber['group_uid'].nunique()
            ),
            'selected_rubber_pixels': int(len(full_rubber)),
        }
        full_path = ablation_dir / f'full_fold_{outer_fold_id}.json'
        if full_path.exists():
            full_record = load_json(full_path)
            print('  full condition: reused')
        else:
            full_model = fit_svm(
                train_frame, S2_FEATURES, FINAL_PARAMS
            )
            metrics, pixel_predictions, polygon_predictions = evaluate_model(
                full_model, validation_frame, S2_FEATURES
            )
            full_record = ablation_metrics_record(
                'full', outer_fold_id, -1, metrics,
                pixel_predictions, polygon_predictions, full_audit,
            )
            atomic_write_json(full_record, full_path)
            del full_model, pixel_predictions, polygon_predictions
            gc.collect()
            print('  full condition: completed')
        ablation_records.append(full_record)

        for repeat_id in range(RUBBER_ABLATION_REPEATS):
            reduced_path = (
                ablation_dir
                / f'reduced_fold_{outer_fold_id}_repeat_{repeat_id}.json'
            )
            if reduced_path.exists():
                reduced_record = load_json(reduced_path)
                print(f'  reduced repeat {repeat_id}: reused')
            else:
                reduced_seed = (
                    RANDOM_SEED + 50_000
                    + outer_fold_id * 1000 + repeat_id
                )
                reduced_train, reduced_audit = (
                    select_reduced_rubber_training(
                        train_frame, reduced_seed
                    )
                )
                reduced_model = fit_svm(
                    reduced_train, S2_FEATURES, FINAL_PARAMS
                )
                metrics, pixel_predictions, polygon_predictions = evaluate_model(
                    reduced_model, validation_frame, S2_FEATURES
                )
                reduced_record = ablation_metrics_record(
                    'rubber_reduced_to_bentong_scale',
                    outer_fold_id,
                    repeat_id,
                    metrics,
                    pixel_predictions,
                    polygon_predictions,
                    reduced_audit,
                )
                atomic_write_json(reduced_record, reduced_path)
                del (
                    reduced_train, reduced_model,
                    pixel_predictions, polygon_predictions,
                )
                gc.collect()
                print(f'  reduced repeat {repeat_id}: completed')
            ablation_records.append(reduced_record)

    ablation_results = pd.DataFrame(ablation_records)
    atomic_write_csv(
        ablation_results,
        TABLE_DIR / 'rubber_sample_size_ablation_records.csv',
    )

    ablation_metric_columns = [
        'pixel_rubber_f1',
        'polygon_rubber_f1',
        'pixel_durian_f1',
        'polygon_durian_f1',
        'pixel_macro_f1',
        'polygon_macro_f1',
        'polygon_rubber_to_durian_rate',
        'polygon_durian_to_rubber_rate',
    ]
    ablation_summary = (
        ablation_results.groupby('condition')[ablation_metric_columns]
        .agg(['mean', 'std', 'min', 'max'])
    )
    ablation_summary.columns = [
        f'{metric}_{stat}'
        for metric, stat in ablation_summary.columns
    ]
    ablation_summary = ablation_summary.reset_index()
    atomic_write_csv(
        ablation_summary,
        TABLE_DIR / 'rubber_sample_size_ablation_summary.csv',
    )

    full_by_fold = (
        ablation_results.loc[ablation_results['condition'] == 'full']
        .set_index('outer_fold_id')
    )
    reduced_rows = ablation_results.loc[
        ablation_results['condition'] != 'full'
    ].copy()
    for metric in ablation_metric_columns:
        reduced_rows[f'delta_{metric}_reduced_minus_full'] = (
            reduced_rows[metric].to_numpy()
            - reduced_rows['outer_fold_id'].map(full_by_fold[metric]).to_numpy()
        )
    atomic_write_csv(
        reduced_rows,
        TABLE_DIR / 'rubber_sample_size_ablation_paired_differences.csv',
    )

    plt.figure(figsize=(10, 5))
    sns.boxplot(
        data=ablation_results,
        x='condition',
        y='polygon_rubber_f1',
    )
    sns.stripplot(
        data=ablation_results,
        x='condition',
        y='polygon_rubber_f1',
        color='black',
        alpha=0.65,
    )
    plt.ylim(0, 1)
    plt.title('Rubber sample-size experiment — polygon Rubber F1')
    plt.xlabel('Training condition')
    plt.ylabel('Polygon Rubber F1')
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / 'rubber_sample_size_ablation_polygon_f1.png',
        dpi=180,
        bbox_inches='tight',
    )
    plt.show()

    display(ablation_summary)
    display(reduced_rows[[
        'outer_fold_id',
        'repeat_id',
        'selected_rubber_groups',
        'selected_rubber_polygons',
        'delta_polygon_rubber_f1_reduced_minus_full',
        'delta_polygon_rubber_to_durian_rate_reduced_minus_full',
    ]])
else:
    ablation_summary = None
    print('Rubber ablation skipped by RUN_RUBBER_ABLATION=False.')

## 10. 保存 manifest、README 和完成标记

In [ ]:
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'workflow': WORKFLOW_VERSION,
    'input_csv': str(INPUT_CSV),
    'input_csv_sha256': INPUT_SHA256,
    'output_directory': str(OUTPUT_DIR),
    'target_column_in_source_csv': 'label_id',
    'note_on_source_class_id': (
        'The source class_id stores spatial_group_no and is not the '
        'seven-class target.'
    ),
    'class_to_id': CLASS_TO_ID,
    'predictor_bands': S2_FEATURES,
    'feature_set': 'S2',
    'max_pixels_per_sample': MAX_PIXELS_PER_SAMPLE,
    'random_seed': RANDOM_SEED,
    'n_splits': N_SPLITS,
    'inner_splits': INNER_SPLITS,
    'svm_candidates': SVM_CANDIDATES,
    'selection_order': SELECTION_METRICS,
    'final_candidate_id': FINAL_CANDIDATE_ID,
    'final_params': FINAL_PARAMS,
    'model_rows': int(len(model_df)),
    'sample_count': int(sample_table['sample_uid'].nunique()),
    'group_count': int(sample_table['group_uid'].nunique()),
    'nested_oof_metrics': nested_overall_metrics,
    'rubber_ablation': {
        'enabled': RUN_RUBBER_ABLATION,
        'repeats': RUBBER_ABLATION_REPEATS,
        'bentong_reference_total_groups': BENTONG_RUBBER_GROUPS_TOTAL,
        'bentong_reference_total_polygons': BENTONG_RUBBER_POLYGONS_TOTAL,
        'target_groups_per_outer_training': TARGET_RUBBER_GROUPS_TRAIN,
        'target_polygons_per_outer_training': TARGET_RUBBER_POLYGONS_TRAIN,
        'interpretation': (
            'Pahang-vs-Bentong internal metrics are descriptive. The paired '
            'full-vs-reduced Pahang experiment more directly tests the '
            'effect of Rubber training sample diversity and quantity.'
        ),
    },
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'joblib': joblib.__version__,
    },
}
atomic_write_json(manifest, METADATA_DIR / 'model_manifest.json')

readme_text = f'''Pahang S2 SVM grouped validation
=================================

Input CSV: {INPUT_CSV}
Output directory: {OUTPUT_DIR}
Feature set: S2 ({len(S2_FEATURES)} predictors)
Outer/inner folds: {N_SPLITS}/{INNER_SPLITS}
Final candidate: {FINAL_CANDIDATE_ID}

Formal performance estimate
---------------------------
Use metadata/nested_overall_metrics.json and the OOF reports/confusion
matrices. Do not report the final full-data training fit as validation.

Rubber hypothesis
-----------------
The Bentong-vs-Pahang table is descriptive only. The direct test is the
paired full-vs-reduced Pahang experiment in:
tables/rubber_sample_size_ablation_records.csv
tables/rubber_sample_size_ablation_summary.csv
tables/rubber_sample_size_ablation_paired_differences.csv

Main outputs
------------
tables/pahang_raw_class_summary.csv
tables/pahang_model_class_summary.csv
tables/fold_assignments.csv
tables/training_rows_used.csv
tables/nested_inner_candidate_scores.csv
tables/nested_outer_fold_metrics.csv
tables/nested_selected_candidates.csv
tables/oof_pixel_predictions.csv
tables/oof_polygon_predictions.csv
tables/oof_pixel_classification_report.csv
tables/oof_polygon_classification_report.csv
tables/final_candidate_scores.csv
models/pahang_svm_final_bundle.joblib
metadata/model_manifest.json
'''
with open(OUTPUT_DIR / 'README.txt', 'w', encoding='utf-8') as file:
    file.write(readme_text)

with open(METADATA_DIR / 'COMPLETED.txt', 'w', encoding='utf-8') as file:
    file.write(datetime.now(timezone.utc).isoformat() + '\n')

print('Pahang S2 SVM workflow completed.')
print('Output directory:', OUTPUT_DIR)
print('Formal nested OOF metrics:')
print(json.dumps(nested_overall_metrics, indent=2))